# Ocean Alkalinity Enhancement (OAE) analysis

This notebook analyzes the Iceland3_MARBL_2024 OAE experiments in Hvalfjörður: how the imposed alkalinity (and associated carbon) perturbation evolves, and how efficiently it removes atmospheric CO₂.

It reads only the intermediate files that `OAE_ensemble.ipynb` and `mixed_layer_depth.ipynb` write to `intermediate_dir`, so run those first. The one exception is the last appendix cell, which plots a CDR forcing (model input) file.

| § | Figure(s) | Reads (in `intermediate_dir`) |
|---|---|---|
| 1 | Added-ALK inventory and its fraction in the mixed layer; DIC created vs. retained (CDR1–6) | `Iceland3_{ens}_mld_metrics.nc`, `CDR{n}_cdr_metrics.nc` |
| 2 | 0–96 h mean maps and ensemble extrema, July and November | `Iceland3_1m_time0_96_mean_{season}.nc`, `Iceland3_CDR_extrema_metrics_{season}.nc` |
| 3 | Maximum pH vs. background carbonate states (PyCO2SYS) | `Iceland3_CDR_extrema_metrics_July.nc` |
| 4 | Cumulative CO₂ flux and DIC inventory at 180 h; CDR efficiency | `Iceland3_CDR_cstar_cumsum_fields_{season}.nc`, `CDR{n}_cdr_metrics.nc` |
| 5 | Release snapshots of ALK, pH and surface pCO₂ (saved as PNG) | `Iceland3_release_snapshots_{season}.nc` |
| A | Diagnostics: CDR metrics in absolute time, time coverage, CDR forcing | `CDR{n}_cdr_metrics.nc`, `ds_cstar{n}_regridded.nc` |

`{ens}` is CDR1…CDR6, `{n}` is 1…6 and `{season}` is July or November. Saved figures go to `fig_dir`.

## Configuration

Location of the intermediate files and figures, ensemble members and release start times.

In [ ]:
import os

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

intermediate_dir = "/home/x-uheede/R/Intermediate_files/"  # written by OAE_ensemble.ipynb
fig_dir = "."  # saved figures go next to this notebook

seasons = {
    "July": ["CDR1", "CDR2", "CDR3"],
    "November": ["CDR4", "CDR5", "CDR6"],
}
ensembles = [ens for members in seasons.values() for ens in members]

# Release start of each member in hours after its first model record (aligns the ALK inventory in §1)
release_start_hours = {"CDR1": 36.0, "CDR2": 70.0, "CDR3": 70.0, "CDR4": 48.0, "CDR5": 54.0, "CDR6": 48.0}


def intermediate(name):
    """Path of an intermediate file; stops with a clear message if it has not been written."""
    path = os.path.join(intermediate_dir, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found: run the matching section of OAE_ensemble.ipynb")
    return path

## 1. Added alkalinity and its fraction in the mixed layer

Each member is aligned at its release start (`release_start_hours`), where the added-ALK inventory is baseline-subtracted. The fraction of added ALK above the seasonal mixed-layer depth is smoothed with a 10-h running mean.

The second figure multiplies the flux- and DIC-based CDR efficiencies by the added ALK, held at its maximum. This gives the DIC created by air–sea exchange and the DIC retained in the domain; the difference has been exported out of the domain.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# ====================================================
# File Paths & Configuration
# ====================================================
file_paths = [intermediate(f"Iceland3_{ens}_mld_metrics.nc") for ens in ensembles]

# Specific requested color palette: blue, green, orange, purple, brown, pink
color_palette = [
    "#1f77b4",  # Blue
    "#2ca02c",  # Green
    "#ff7f0e",  # Orange
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
]

data = {}

# ====================================================
# Data Loading & Processing
# ====================================================
for path in file_paths:
    ds = xr.open_dataset(path)
    ens_id = ds.attrs.get("ensemble", os.path.basename(path).split("_")[1])

    if "total_alk" not in ds or "mld_alk_fraction" not in ds:
        raise KeyError(
            f"Required variables ('total_alk', 'mld_alk_fraction') missing in {path}."
        )

    tot_alk = ds["total_alk"].load()
    alk_frac = ds["mld_alk_fraction"].load()

    # 1. Convert time coordinate to numeric floating-point hours from file start
    raw_time = ds["time"]

    if np.issubdtype(raw_time.dtype, np.datetime64):
        # Convert datetime64 to elapsed hours float
        time_hours = (raw_time - raw_time.isel(time=0)) / np.timedelta64(1, "h")
    elif np.issubdtype(raw_time.dtype, np.timedelta64):
        # Convert timedelta64 to hours
        time_hours = (raw_time - raw_time.isel(time=0)) / np.timedelta64(1, "h")
    else:
        # If numeric: if values are small (<100), assume days and convert to hours
        raw_vals = raw_time.values - raw_time.values[0]
        time_hours = xr.DataArray(
            raw_vals * 24.0 if raw_vals.max() < 100 else raw_vals,
            dims=raw_time.dims,
        )

    # 2. Find release start index based on ensemble start time
    target_hour = release_start_hours.get(ens_id, 0.0)
    release_start_idx = int(np.argmin(np.abs(time_hours.values - target_hour)))

    # 3. Compute relative time in hours (t_rel = 0 at release start)
    time_rel = time_hours - time_hours.isel(time=release_start_idx)

    # 4. Baseline-subtract total alkalinity at release start
    alk_added = tot_alk - tot_alk.isel(time=release_start_idx)

    # 5. Smooth upper-layer ALK fraction (10-hr rolling mean)
    alk_frac_smooth = alk_frac.rolling(
        time=10, center=True, min_periods=1
    ).mean()

    # Convert to pure NumPy arrays to avoid xarray metadata plotting bugs
    data[ens_id] = {
        "time_rel": np.asarray(time_rel.values, dtype=float),
        "alk_added": np.asarray(alk_added.values, dtype=float),
        "alk_frac_smooth": np.asarray(alk_frac_smooth.values, dtype=float),
    }
    ds.close()

# Map sorted ensembles to the requested colors
sorted_keys = sorted(data.keys())
color_map = {
    ens_id: color_palette[i % len(color_palette)]
    for i, ens_id in enumerate(sorted_keys)
}

# ====================================================
# Plotting - 2 Panel Figure (Wider / Elongated)
# ====================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# --- Panel 1: Total Added Alkalinity Inventory ---
for ens_id in sorted_keys:
    t_rel = data[ens_id]["time_rel"]
    alk_added = data[ens_id]["alk_added"]

    mask = t_rel >= 0
    ax1.plot(
        t_rel[mask],
        alk_added[mask],
        label=f"{ens_id}",
        color=color_map[ens_id],
        linewidth=1.8,
    )

ax1.set_ylabel(
    "Added Alkalinity (moles)", fontsize=10, fontweight="bold"
)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.axvline(0, color="black", linestyle="--", alpha=0.7, label="Release Start (t=0)")
ax1.axvline(
    96, color="gray", linestyle=":", alpha=0.7, label="Release End (t=96h)"
)
ax1.set_title(
    "Total Added Alkalinity Over Time",
    fontsize=11,
    fontweight="bold",
)
ax1.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=True)

# --- Panel 2: 10-Hour Running Mean Upper ALK Fraction ---
for ens_id in sorted_keys:
    t_rel = data[ens_id]["time_rel"]
    alk_frac_smooth = data[ens_id]["alk_frac_smooth"]

    mask = t_rel >= 0
    ax2.plot(
        t_rel[mask],
        alk_frac_smooth[mask],
        label=f"{ens_id} (10-hr mean)",
        color=color_map[ens_id],
        linewidth=1.8,
    )

ax2.set_xlabel("Relative Time (hours since release start)", fontsize=10)
ax2.set_ylabel(
    "Fraction of ALK in mixed layer",
    fontsize=10,
    fontweight="bold",
)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.axvline(0, color="black", linestyle="--", alpha=0.7)
ax2.axvline(96, color="gray", linestyle=":", alpha=0.7)

# Set x-axis limits to 0 - 185 hours
ax2.set_xlim(-5, 185)

ax2.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=True,
)

ax2.set_title(
    "Fraction of Added Alkalinity Retained in Mixed Layer",
    fontsize=11,
    fontweight="bold",
)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "mld_alk_total_unnormalized_2panel.jpeg"), format="jpeg", dpi=300)
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# ====================================================
# Configuration & File Paths
# ====================================================
# Members, release start hours and intermediate_dir: see the configuration cell

plot_data = {}

# ====================================================
# Data Processing Loop
# ====================================================
for ens in ensembles:
    mld_file = intermediate(f"Iceland3_{ens}_mld_metrics.nc")
    cdr_file = intermediate(f"{ens}_cdr_metrics.nc")

    ds_mld = xr.open_dataset(mld_file)
    ds_cdr = xr.open_dataset(cdr_file)

    # 1. Load ALK inventory and CDR efficiency arrays
    tot_alk = ds_mld["total_alk"].load()

    # Efficiency from the air-sea flux difference and from the DIC difference
    eta_flux = ds_cdr["cdr_efficiency"].load()
    eta_dic = ds_cdr["cdr_efficiency_from_dic"].load()

    # 2. Convert MLD time coordinate to relative hours from release start
    raw_time = ds_mld["time"]
    if np.issubdtype(raw_time.dtype, np.datetime64):
        time_hours = (raw_time - raw_time.isel(time=0)) / np.timedelta64(1, "h")
    elif np.issubdtype(raw_time.dtype, np.timedelta64):
        time_hours = (raw_time - raw_time.isel(time=0)) / np.timedelta64(1, "h")
    else:
        raw_vals = raw_time.values - raw_time.values[0]
        time_hours = xr.DataArray(
            raw_vals * 24.0 if raw_vals.max() < 100 else raw_vals,
            dims=raw_time.dims,
        )

    # Align release start index (t_rel = 0)
    target_hour = release_start_hours[ens]
    release_start_idx = int(np.argmin(np.abs(time_hours.values - target_hour)))
    time_rel = time_hours - time_hours.isel(time=release_start_idx)

    # 3. Compute baseline-subtracted added alkalinity
    alk_added = tot_alk - tot_alk.isel(time=release_start_idx)
    alk_added_vals = np.maximum(0.0, np.asarray(alk_added.values, dtype=float))

    # 4. Freeze alkalinity inventory at its peak (holding max value for subsequent timesteps)
    alk_held = np.maximum.accumulate(alk_added_vals)

    # 5. Extract raw numpy arrays safely
    eta_flux_vals = np.nan_to_num(np.asarray(eta_flux.values, dtype=float), nan=0.0)
    eta_dic_vals = np.nan_to_num(np.asarray(eta_dic.values, dtype=float), nan=0.0)

    # Match array lengths across datasets
    min_len = min(len(alk_held), len(eta_flux_vals), len(eta_dic_vals))

    t_rel_vals = np.asarray(time_rel.values, dtype=float)[:min_len]
    alk_held_slice = alk_held[:min_len]
    eta_flux_slice = eta_flux_vals[:min_len]
    eta_dic_slice = eta_dic_vals[:min_len]

    # 6. Calculate total DIC moles created (flux) vs. retained (inventory)
    dic_from_flux = eta_flux_slice * alk_held_slice
    dic_from_inventory = eta_dic_slice * alk_held_slice

    plot_data[ens] = {
        "time_rel": t_rel_vals,
        "dic_flux": dic_from_flux,
        "dic_inventory": dic_from_inventory,
        "alk_held": alk_held_slice,
    }

    ds_mld.close()
    ds_cdr.close()

# ====================================================
# 6-Panel Figure Generation
# ====================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i, ens in enumerate(ensembles):
    ax = axes[i]

    if ens not in plot_data:
        ax.set_title(f"{ens} (Data Missing)")
        continue

    t_rel = plot_data[ens]["time_rel"]
    dic_flux = plot_data[ens]["dic_flux"]
    dic_inv = plot_data[ens]["dic_inventory"]

    mask = t_rel >= 0

    # Plot DIC curves
    ax.plot(
        t_rel[mask],
        dic_flux[mask],
        color="crimson",
        linewidth=2.0,
        label="DIC Created (from Air-Sea Flux Difference)",
    )
    ax.plot(
        t_rel[mask],
        dic_inv[mask],
        color="navy",
        linewidth=2.0,
        label="DIC Retained (from Domain DIC Inventory)",
    )

    # Shade difference representing carbon exported out of domain
    ax.fill_between(
        t_rel[mask],
        dic_inv[mask],
        dic_flux[mask],
        where=(dic_flux[mask] >= dic_inv[mask]),
        color="orange",
        alpha=0.35,
        label="Exported DIC (Out of Domain)",
    )

    # Add reference lines
    ax.axvline(
        0, color="black", linestyle="--", alpha=0.6, label="Release Start (t=0)"
    )
    ax.axvline(96, color="gray", linestyle=":", alpha=0.6, label="Release End (t=96h)")

    ax.set_title(f"Ensemble Experiment: {ens}", fontsize=12, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)

    # Labeling
    if i in [0, 3]:
        ax.set_ylabel(
            "Cumulative DIC Created / Retained (moles)",
            fontsize=11,
            fontweight="bold",
        )
    if i >= 3:
        ax.set_xlabel(
            "Relative Time (hours since release)",
            fontsize=11,
            fontweight="bold",
        )

# Set common x-limits
max_t = max(plot_data[ens]["time_rel"].max() for ens in plot_data if ens in plot_data)
ax.set_xlim(-5, min(185, max_t))

# Shared legend placed cleanly above subplots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=4,
    frameon=True,
    fontsize=11,
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(
    os.path.join(fig_dir, "CDR_6panel_DIC_moles_and_export.jpeg"),
    format="jpeg",
    dpi=300,
)
plt.show()

## 2. Surface fields and ensemble extrema

One figure per season:
- **Maps:** ensemble means of the 0–96 h mean surface pCO₂ anomaly and of pH, ALK and DIC anomalies at 1 m, with detectability contours in white. The contour levels differ between seasons, as in the submitted figures.
- **Time series:** the domain extrema of each member, the ensemble range (shading) and the ensemble mean (black), vs. hours since release start.

The first cell defines the map scale bar.

In [ ]:
from pyproj import Geod
import matplotlib.pyplot as plt

def add_scalebar(ax, length_km, location=(0.1, 0.05), linewidth=3):
    """
    Add a scale bar to a Cartopy map.

    Parameters
    ----------
    ax : cartopy.mpl.geoaxes.GeoAxes
        Map axes
    length_km : float
        Length of scale bar in kilometers
    location : tuple
        (x_frac, y_frac) position in axes coordinates
    linewidth : int
        Line width
    """

    geod = Geod(ellps="WGS84")

    # Get current map extent in PlateCarree
    lon_min, lon_max, lat_min, lat_max = ax.get_extent(ccrs.PlateCarree())

    # Starting point
    lon_start = lon_min + location[0] * (lon_max - lon_min)
    lat_start = lat_min + location[1] * (lat_max - lat_min)

    # Compute endpoint using geodesic
    lon_end, lat_end, _ = geod.fwd(
        lon_start, lat_start,
        az=90,              # eastward
        dist=length_km * 1000
    )

    # Draw scale bar
    ax.plot(
        [lon_start, lon_end],
        [lat_start, lat_start],
        transform=ccrs.PlateCarree(),
        color="black",
        linewidth=linewidth
    )

    # Label
    ax.text(
        (lon_start + lon_end) / 2,
        lat_start + 0.002,
        f"{length_km} km",
        transform=ccrs.PlateCarree(),
        ha="center",
        va="bottom",
        fontsize=9
    )

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

proj = ccrs.Mercator()


# ===============================
# Load NetCDF files
# ===============================

season = "July"
ds_mean = xr.open_dataset(intermediate(f"Iceland3_1m_time0_96_mean_{season}.nc"))
ds_metrics = xr.open_dataset(intermediate(f"Iceland3_CDR_extrema_metrics_{season}.nc"))
# ===============================
# Extract variables
# ===============================

FG_snapshot = ds_mean["pCO2SURF_anom"]
pH_surface = ds_mean["pH"]
ALK_anom_surface = ds_mean["ALK_anom"]
DIC_anom_surface = ds_mean["DIC_anom"]

FG_min = ds_metrics["min_pCO2SURF_anom"]
pH_max = ds_metrics["max_pH"]
ALK_max = ds_metrics["max_ALK_anom"]
DIC_max = ds_metrics["max_DIC_anom"]

time_rel = ds_metrics.time_rel
time_rel = ds_metrics.time_rel

time_label = "time mean across 0–96 h"

# -------------------------------
# Hvalfjörður extent
# -------------------------------
lon_min, lon_max = -21.6, -21.37
lat_min, lat_max = 64.34, 64.42

# Convert if grid is 0–360
#if ds_bgc_dia_ens.lon.max() > 180:
lon_min = lon_min % 360
lon_max = lon_max % 360

# ===============================
# Layout (2x2 maps + 2x2 TS)
# ===============================
fig = plt.figure(figsize=(16, 18))

gs = fig.add_gridspec(
    nrows=6,
    ncols=4,
    height_ratios=[3.5, 3.5, 1.2, 1.2, 1.2, 1.2],
    hspace=0.4,
    wspace=0.25
)

ax_c1 = fig.add_subplot(gs[0, 0:2], projection=proj)
ax_c2 = fig.add_subplot(gs[0, 2:4], projection=proj)
ax_c3 = fig.add_subplot(gs[1, 0:2], projection=proj)
ax_c4 = fig.add_subplot(gs[1, 2:4], projection=proj)

ax_ts1 = fig.add_subplot(gs[2:4, 0:2])
ax_ts2 = fig.add_subplot(gs[2:4, 2:4])
ax_ts3 = fig.add_subplot(gs[4:6, 0:2])
ax_ts4 = fig.add_subplot(gs[4:6, 2:4])

panel_labels = ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)"]

def plot_map(ax, data, title, panel_label,stipple_level=None):

    ax.set_extent(
        [lon_min, lon_max, lat_min, lat_max],
        crs=ccrs.PlateCarree()
    )

    mean_field = data.mean("ens")
    std_field = data.std("ens")

    # -----------------------------
    # filled map
    # -----------------------------
    cf = ax.contourf(
        data.lon,
        data.lat,
        mean_field,
        levels=20,
        transform=ccrs.PlateCarree()
    )
    
    # -----------------------------
    # stippled contour
    # -----------------------------
    if stipple_level is not None:
        ax.contour(
            data.lon,
            data.lat,
            mean_field,
            levels=[stipple_level],
            colors="white",
            linewidths=0.8,
            hatches=["-"],
            transform=ccrs.PlateCarree()
        )
        ax.contour(
            data.lon,
            data.lat,
            mean_field,
            levels=[stipple_level*2],
            colors="white",
            linewidths=0.8,
            hatches=["-"],
            transform=ccrs.PlateCarree()
        )

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        linestyle="--",
        alpha=0.5
    )

    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(title)

    ax.text(
        0.02,
        0.95,
        panel_label,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

    add_scalebar(ax, 1)

    return cf
# ===============================
# Map plots
# ===============================

cf1 = plot_map(
    ax_c1,
    FG_snapshot,
    f"Surface pCO$_2$ anomaly\n({time_label})",
    panel_labels[0],
    stipple_level=-9.2   # pCO2
)

cf2 = plot_map(
    ax_c2,
    pH_surface,
    f"pH at 1m depth\n({time_label})",
    panel_labels[1],
    stipple_level=8.12  # pH
)

cf2 = plot_map(
    ax_c2,
    pH_surface,
    f"pH at 1m depth\n({time_label})",
    panel_labels[1],
    stipple_level=8.14# pH
)


cf3 = plot_map(
    ax_c3,
    ALK_anom_surface,
    f"ALK anomaly at 1m depth\n({time_label})",
    panel_labels[2],
    stipple_level=5   # ALK
)

cf4 = plot_map(
    ax_c4,
    DIC_anom_surface,
    f"DIC anomaly at 1m depth\n({time_label})",
    panel_labels[3],
    stipple_level=None  # no detectability contour
)

cb1 = fig.colorbar(cf1, ax=ax_c1, shrink=0.8)
cb1.set_label("ppmv")

cb2 = fig.colorbar(cf2, ax=ax_c2, shrink=0.8)
cb2.set_label("pH")

cb3 = fig.colorbar(cf3, ax=ax_c3, shrink=0.8)
cb3.set_label("mmol m$^{-3}$")

cb4 = fig.colorbar(cf4, ax=ax_c4, shrink=0.8)
cb4.set_label("mmol m$^{-3}$")

# ===============================
# Time series
# ===============================

ts_axes = [ax_ts1, ax_ts2, ax_ts3, ax_ts4]

ts_data = [
    (time_rel, FG_min, "Min pCO2 anomaly", "ppmv"),
    (time_rel, pH_max, "Max pH", "pH"),
    (time_rel, ALK_max, "Max ALK anomaly", "mmol m$^{-3}$"),
    (time_rel, DIC_max, "Max DIC anomaly", "mmol m$^{-3}$")
]
for i, ax in enumerate(ts_axes):

    t, data, ylabel, units = ts_data[i]

    y_min = data.min("ens")
    y_max = data.max("ens")
    y_mean = data.mean("ens")

    # -----------------------
    # ensemble members
    # -----------------------
    for j, ens in enumerate(data.ens.values):
        
        # Only label for first subplot (pCO2)
        if i == 0:
            label = f"CDR{j+1}"
        else:
            label = None

        ax.plot(
            t,
            data.sel(ens=ens),
            linewidth=1.8,
            alpha=0.7,
            label=label
        )

    # -----------------------
    # ensemble spread
    # -----------------------
    ax.fill_between(
        t,
        y_min,
        y_max,
        alpha=0.3
    )

    # -----------------------
    # ensemble mean
    # -----------------------
    ax.plot(
        t,
        y_mean,
        color="black",
        linewidth=1.8,
        label="Mean" if i == 0 else None
    )

    # -----------------------
    # legend ONLY for pCO2
    # -----------------------
    if i == 0:
        ax.legend(fontsize=11)

    ax.set_ylabel(units, fontsize=13)
    ax.set_title(f"{ylabel} over time", fontsize=14)

    ax.tick_params(axis='both', labelsize=12)

    ax.axvline(0, linestyle=":", linewidth=1.5, color="black")
    ax.axvline(96, linestyle=":", linewidth=1.5, color="black")

    ax.grid(True, alpha=0.3)

    ax.text(
        0.02,
        0.95,
        panel_labels[i+4],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_horizontalalignment("right")

    if i in [0, 1]:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", which="both", length=0)

    ax.set_ylabel(units, fontsize=13)
    ax.set_title(f"{ylabel} over time", fontsize=14)

    ax.tick_params(axis='both', labelsize=12)

    ax.axvline(0, linestyle=":", linewidth=1.5, color="black")
    ax.axvline(96, linestyle=":", linewidth=1.5, color="black")

    ax.grid(True, alpha=0.3)

    ax.text(
        0.02,
        0.95,
        panel_labels[i+4],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_horizontalalignment("right")

    if i in [0, 1]:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", which="both", length=0)
        
# Add x-axis label for bottom row
ax_ts3.set_xlabel("Time (hours after release)",fontsize=12)
ax_ts4.set_xlabel("Time (hours after release)",fontsize=12)
plt.show()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

proj = ccrs.Mercator()


# ===============================
# Load NetCDF files
# ===============================

season = "November"
ds_mean = xr.open_dataset(intermediate(f"Iceland3_1m_time0_96_mean_{season}.nc"))
ds_metrics = xr.open_dataset(intermediate(f"Iceland3_CDR_extrema_metrics_{season}.nc"))
# ===============================
# Extract variables
# ===============================

FG_snapshot = ds_mean["pCO2SURF_anom"]
pH_surface = ds_mean["pH"]
ALK_anom_surface = ds_mean["ALK_anom"]
DIC_anom_surface = ds_mean["DIC_anom"]

FG_min = ds_metrics["min_pCO2SURF_anom"]
pH_max = ds_metrics["max_pH"]
ALK_max = ds_metrics["max_ALK_anom"]
DIC_max = ds_metrics["max_DIC_anom"]

time_rel = ds_metrics.time_rel
time_rel = ds_metrics.time_rel

time_label = "time mean across 0–96 h"

# -------------------------------
# Hvalfjörður extent
# -------------------------------
lon_min, lon_max = -21.6, -21.37
lat_min, lat_max = 64.34, 64.42

# Convert if grid is 0–360
#if ds_bgc_dia_ens.lon.max() > 180:
lon_min = lon_min % 360
lon_max = lon_max % 360

# ===============================
# Layout (2x2 maps + 2x2 TS)
# ===============================
fig = plt.figure(figsize=(16, 18))

gs = fig.add_gridspec(
    nrows=6,
    ncols=4,
    height_ratios=[3.5, 3.5, 1.2, 1.2, 1.2, 1.2],
    hspace=0.4,
    wspace=0.25
)

ax_c1 = fig.add_subplot(gs[0, 0:2], projection=proj)
ax_c2 = fig.add_subplot(gs[0, 2:4], projection=proj)
ax_c3 = fig.add_subplot(gs[1, 0:2], projection=proj)
ax_c4 = fig.add_subplot(gs[1, 2:4], projection=proj)

ax_ts1 = fig.add_subplot(gs[2:4, 0:2])
ax_ts2 = fig.add_subplot(gs[2:4, 2:4])
ax_ts3 = fig.add_subplot(gs[4:6, 0:2])
ax_ts4 = fig.add_subplot(gs[4:6, 2:4])

panel_labels = ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)"]

def plot_map(ax, data, title, panel_label,stipple_level=None):

    ax.set_extent(
        [lon_min, lon_max, lat_min, lat_max],
        crs=ccrs.PlateCarree()
    )

    mean_field = data.mean("ens")
    std_field = data.std("ens")

    # -----------------------------
    # filled map
    # -----------------------------
    cf = ax.contourf(
        data.lon,
        data.lat,
        mean_field,
        levels=20,
        transform=ccrs.PlateCarree()
    )
    
    # -----------------------------
    # stippled contour
    # -----------------------------
    if stipple_level is not None:
        ax.contour(
            data.lon,
            data.lat,
            mean_field,
            levels=[stipple_level],
            colors="white",
            linewidths=0.8,
            hatches=["-"],
            transform=ccrs.PlateCarree()
        )
        ax.contour(
            data.lon,
            data.lat,
            mean_field,
            levels=[stipple_level*2],
            colors="white",
            linewidths=0.8,
            hatches=["-"],
            transform=ccrs.PlateCarree()
        )

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        linestyle="--",
        alpha=0.5
    )

    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(title)

    ax.text(
        0.02,
        0.95,
        panel_label,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

    add_scalebar(ax, 1)

    return cf
# ===============================
# Map plots
# ===============================

cf1 = plot_map(
    ax_c1,
    FG_snapshot,
    f"Surface pCO$_2$ anomaly\n({time_label})",
    panel_labels[0],
    stipple_level=-9.2   # pCO2
)

cf2 = plot_map(
    ax_c2,
    pH_surface,
    f"pH at 1m depth\n({time_label})",
    panel_labels[1],
    stipple_level=8.15  # pH
)

cf2 = plot_map(
    ax_c2,
    pH_surface,
    f"pH at 1m depth\n({time_label})",
    panel_labels[1],
    stipple_level=8.18# pH
)


cf3 = plot_map(
    ax_c3,
    ALK_anom_surface,
    f"ALK anomaly at 1m depth\n({time_label})",
    panel_labels[2],
    stipple_level=6.1   # ALK
)

cf4 = plot_map(
    ax_c4,
    DIC_anom_surface,
    f"DIC anomaly at 1m depth\n({time_label})",
    panel_labels[3],
    stipple_level=None  # no detectability contour
)

cb1 = fig.colorbar(cf1, ax=ax_c1, shrink=0.8)
cb1.set_label("ppmv")

cb2 = fig.colorbar(cf2, ax=ax_c2, shrink=0.8)
cb2.set_label("pH")

cb3 = fig.colorbar(cf3, ax=ax_c3, shrink=0.8)
cb3.set_label("mmol m$^{-3}$")

cb4 = fig.colorbar(cf4, ax=ax_c4, shrink=0.8)
cb4.set_label("mmol m$^{-3}$")

# ===============================
# Time series
# ===============================

ts_axes = [ax_ts1, ax_ts2, ax_ts3, ax_ts4]

ts_data = [
    (time_rel, FG_min, "Min pCO2 anomaly", "ppmv"),
    (time_rel, pH_max, "Max pH", "pH"),
    (time_rel, ALK_max, "Max ALK anomaly", "mmol m$^{-3}$"),
    (time_rel, DIC_max, "Max DIC anomaly", "mmol m$^{-3}$")
]
# Get default color cycle
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
n_colors = len(colors)
for i, ax in enumerate(ts_axes):

    t, data, ylabel, units = ts_data[i]

    y_min = data.min("ens")
    y_max = data.max("ens")
    y_mean = data.mean("ens")

    # -----------------------
    # ensemble members
    # -----------------------
    for j, ens in enumerate(data.ens.values):
    # Shift color index by +3
        color = colors[(j + 3) % n_colors]
        
        # Only label for first subplot (pCO2)
        if i == 0:
            label = f"CDR{j+4}"
        else:
            label = None

        ax.plot(
            t,
            data.sel(ens=ens),
            linewidth=1.8,
            alpha=0.7,
            label=label,
            color=color
        )

    # -----------------------
    # ensemble spread
    # -----------------------
    ax.fill_between(
        t,
        y_min,
        y_max,
        alpha=0.3
    )

    # -----------------------
    # ensemble mean
    # -----------------------
    ax.plot(
        t,
        y_mean,
        color="black",
        linewidth=1.8,
        label="Mean" if i == 0 else None
    )

    # -----------------------
    # legend ONLY for pCO2
    # -----------------------
    if i == 0:
        ax.legend(fontsize=11)

    ax.set_ylabel(units, fontsize=13)
    ax.set_title(f"{ylabel} over time", fontsize=14)

    ax.tick_params(axis='both', labelsize=12)

    ax.axvline(0, linestyle=":", linewidth=1.5, color="black")
    ax.axvline(96, linestyle=":", linewidth=1.5, color="black")

    ax.grid(True, alpha=0.3)

    ax.text(
        0.02,
        0.95,
        panel_labels[i+4],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_horizontalalignment("right")

    if i in [0, 1]:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", which="both", length=0)

    ax.set_ylabel(units, fontsize=13)
    ax.set_title(f"{ylabel} over time", fontsize=14)

    ax.tick_params(axis='both', labelsize=12)

    ax.axvline(0, linestyle=":", linewidth=1.5, color="black")
    ax.axvline(96, linestyle=":", linewidth=1.5, color="black")

    ax.grid(True, alpha=0.3)

    ax.text(
        0.02,
        0.95,
        panel_labels[i+4],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_horizontalalignment("right")

    if i in [0, 1]:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", which="both", length=0)
        
# Add x-axis label for bottom row
ax_ts3.set_xlabel("Time (hours after release)",fontsize=12)
ax_ts4.set_xlabel("Time (hours after release)",fontsize=12)
plt.show()

## 3. Background carbonate-system sensitivity

PyCO2SYS pH and pCO₂ for the ensemble-mean maximum ALK anomaly added to two background states, observed (scenario 1) and modelled (scenario 2). These are compared with the range of modelled maximum pH.

This uses the **July** ensemble (`background_season`). In the original notebook this figure was run right after the July maps, and the background temperatures (10 and 11.5 °C) are summer values.

In [ ]:
# Background states are applied to the July ensemble (see above)
background_season = "July"
ds_background = xr.open_dataset(intermediate(f"Iceland3_CDR_extrema_metrics_{background_season}.nc"))

ALK_max = ds_background["max_ALK_anom"]
pH_max = ds_background["max_pH"]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import PyCO2SYS as pyco2

max_ALK = ALK_max.mean('ens')+2320 #maximum background alkalinity
# -------------------------------
# 1. Extract max ALK timeseries
# -------------------------------

time = ALK_max['time_rel']

# -------------------------------
# 2. Set background constants
# -------------------------------
BACKGROUND1 = dict(
    ALK = max_ALK+60,     # mmol/m3 
    DIC = 2160.0,     # mmol/m3
    PO4 = 0.3,        ## mmol/m3
    Si = 2,         # mmol/m3
    T = 10,
    P = 0,
    S = 33.75,
)

BACKGROUND2 = dict(
    ALK = max_ALK,     # mmol/m3
    DIC = 2110.0,     # mmol/m3
    PO4 = 0.8,        ## mmol/m3
    Si = 6.8,         # mmol/m3
    T = 11.5,
    P = 0,
    S = 33.75)

import pandas as pd

# -------------------------------
# Collect scenarios
# -------------------------------
scenarios = {"Scenario 1": BACKGROUND1, "Scenario 2": BACKGROUND2}

# -------------------------------
# Build summary table
# -------------------------------
summary_rows = []
for name, params in scenarios.items():
    row = {
        "Scenario": name,
        "ALK (mmol/m3)": params["ALK"].values if isinstance(params["ALK"], xr.DataArray) else params["ALK"],
        "DIC (mmol/m3)": params["DIC"],
        "PO4 (mmol/m3)": params["PO4"],
        "Si (mmol/m3)": params["Si"],
        "T (°C)": params["T"],
        "P (dbar)": params["P"],
        "S (psu)": params["S"]
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df)

In [ ]:
import numpy as np
import PyCO2SYS as pyco2

def get_csys_with_volumetric_inputs(DATA):
    # --------------------------------
    # 1. Extract values
    # --------------------------------
    alk_vol = np.atleast_1d(DATA["ALK"]).astype(float)
    dic_vol = np.atleast_1d(DATA["DIC"]).astype(float)
    po4_vol = np.atleast_1d(DATA["PO4"]).astype(float)
    si_vol  = np.atleast_1d(DATA["Si"]).astype(float)

    # --------------------------------
    # 2. Clean inputs
    # --------------------------------
    alk_vol[alk_vol < 0] = np.nan
    dic_vol[dic_vol < 0] = np.nan

    # --------------------------------
    # 3. Get density
    # --------------------------------
    temp_results = pyco2.sys(
        par1=alk_vol,
        par2=dic_vol,
        par1_type=1,
        par2_type=2,
        salinity=DATA["S"],
        temperature=DATA["T"],
        pressure=DATA["P"]
    )

    rho = temp_results.get("rho", temp_results.get("density"))

    if rho is None:
        print("Warning: Density not found, using approximate seawater density.")
        rho = 1.026

    # --------------------------------
    # 4. Convert mmol/m3 → µmol/kg
    # --------------------------------
    alk_grav = alk_vol / rho
    dic_grav = dic_vol / rho
    po4_grav = po4_vol / rho
    si_grav  = si_vol / rho

    # --------------------------------
    # 5. Final carbonate system
    # --------------------------------
    results = pyco2.sys(
        par1=alk_grav,
        par2=dic_grav,
        par1_type=1,
        par2_type=2,
        salinity=DATA["S"],
        temperature=DATA["T"],
        pressure=DATA["P"],
        total_phosphate=po4_grav,
        total_silicate=si_grav
    )

    return results


# --------------------------------
# Scenario 1
# --------------------------------
bg1 = get_csys_with_volumetric_inputs(BACKGROUND1)
pH_bg1 = bg1["pH_total"]


# --------------------------------
# Scenario 2
# --------------------------------
bg2 = get_csys_with_volumetric_inputs(BACKGROUND2)
pH_bg2 = bg2["pH_total"]


# --------------------------------
# Print statistics
# --------------------------------
print("\nBACKGROUND 1")
print(f"pH values: {len(pH_bg1)}")
print(f"Average pH: {np.nanmean(pH_bg1):.4f}")


print("\nBACKGROUND 2")
print(f"pH values: {len(pH_bg2)}")
print(f"Average pH: {np.nanmean(pH_bg2):.4f}")

In [ ]:
import numpy as np
import PyCO2SYS as pyco2

def get_csys_with_volumetric_inputs(DATA):
    # --------------------------------
    # 1. Extract values
    # --------------------------------
    alk_vol = np.atleast_1d(DATA["ALK"]).astype(float)
    dic_vol = np.atleast_1d(DATA["DIC"]).astype(float)
    po4_vol = np.atleast_1d(DATA["PO4"]).astype(float)
    si_vol  = np.atleast_1d(DATA["Si"]).astype(float)

    # --------------------------------
    # 2. Clean inputs
    # --------------------------------
    alk_vol[alk_vol < 0] = np.nan
    dic_vol[dic_vol < 0] = np.nan

    # --------------------------------
    # 3. Get density
    # --------------------------------
    temp_results = pyco2.sys(
        par1=alk_vol,
        par2=dic_vol,
        par1_type=1,
        par2_type=2,
        salinity=DATA["S"],
        temperature=DATA["T"],
        pressure=DATA["P"]
    )

    rho = temp_results.get("rho", temp_results.get("density"))

    if rho is None:
        print("Warning: Density not found, using approximate seawater density.")
        rho = 1.026

    # --------------------------------
    # 4. Convert mmol/m3 → µmol/kg
    # --------------------------------
    alk_grav = alk_vol / rho
    dic_grav = dic_vol / rho
    po4_grav = po4_vol / rho
    si_grav  = si_vol / rho

    # --------------------------------
    # 5. Final carbonate system
    # --------------------------------
    results = pyco2.sys(
        par1=alk_grav,
        par2=dic_grav,
        par1_type=1,
        par2_type=2,
        salinity=DATA["S"],
        temperature=DATA["T"],
        pressure=DATA["P"],
        total_phosphate=po4_grav,
        total_silicate=si_grav
    )

    return results


# --------------------------------
# Scenario 1
# --------------------------------
bg1 = get_csys_with_volumetric_inputs(BACKGROUND1)
pCO2_bg1 = bg1["pCO2"]

# --------------------------------
# Scenario 2
# --------------------------------
bg2 = get_csys_with_volumetric_inputs(BACKGROUND2)
pCO2_bg2 = bg2["pCO2"]

# --------------------------------
# Print statistics
# --------------------------------
print("\nBACKGROUND 1")

print(f"Average pCO2: {np.nanmean(pCO2_bg1):.2f} µatm")

print("\nBACKGROUND 2")

print(f"Average pCO2: {np.nanmean(pCO2_bg2):.2f} µatm")

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

# --------------------------------
# pH ensemble spread calculation
# --------------------------------
# Note: This assumes pH_max, pH_bg1, and pH_bg2 are already defined 
# in your workspace from your previous processing steps.
time_pH = pH_max.time_rel

pH_min = pH_max.min(dim="ens")
pH_max_env = pH_max.max(dim="ens")
pH_mean = pH_max.mean(dim="ens")

# --------------------------------
# Figure: Single Panel (Panel A)
# --------------------------------
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)

# Plot the ensemble spread (shaded area)
ax.fill_between(
    time_pH,
    pH_min,
    pH_max_env,
    color='green',
    alpha=0.3,
    label='dynamic model ensemble range'
)

# Plot the reference/background states
ax.plot(
    time_pH,
    pH_bg1,
    linestyle='--',
    color='blue',
    label='simulated ALK perturbation + observed background state'
)

ax.plot(
    time_pH,
    pH_bg2,
    linestyle='--',
    color='red',
    label='simulated ALK perturbation + model background state'
)

# Formatting
ax.set_ylabel("pH (total scale)")
ax.set_xlabel("Time (hours after release)")
ax.set_title("Maximum surface pH")
ax.grid(True, linestyle=':', alpha=0.6)

# Release window markers
ax.axvline(0, linestyle=":", color="grey", label='Release Start/End')
ax.axvline(96, linestyle=":", color="grey")

ax.legend(loc='best')

plt.show()

## 4. Cumulative CO₂ uptake and CDR efficiency

- **Maps (a–d):** ensemble means of the cumulative air–sea CO₂ flux anomaly and of the column DIC inventory anomaly 180 h after release, for July and November. They come from `OAE_ensemble.ipynb` §5, aligned at the first air–sea flux anomaly.
- **Panel (e):** the CDR efficiency (∫FG_CO₂ dt / ΔALK) of all members, aligned at their first finite value.

In [ ]:
import cartopy.crs as ccrs
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

lon_min, lon_max = -21.7, -21.37
lat_min, lat_max = 64.34, 64.42

# =========================
# --- LOAD CDR METRICS ---
# =========================
# --- Efficiency alignment for time series panel (e) ---
eff_rel_list = []

for ens in ensembles:
    ds = xr.open_dataset(intermediate(f"{ens}_cdr_metrics.nc"))
    eff = ds["cdr_efficiency"]

    first_valid_index = np.where(np.isfinite(eff.values))[0][0]
    time_rel = np.arange(len(eff)) - first_valid_index

    eff_rel = eff.assign_coords(time_rel=("time", time_rel))
    eff_rel = eff_rel.swap_dims({"time": "time_rel"}).drop_vars("time")

    eff_rel_list.append(eff_rel)

common_time_rel = np.arange(
    max(e.time_rel.min().item() for e in eff_rel_list),
    min(e.time_rel.max().item() for e in eff_rel_list) + 1,
)

eff_aligned = [e.sel(time_rel=common_time_rel) for e in eff_rel_list]

eff_stack = xr.concat(eff_aligned, dim="ens")
eff_stack["ens"] = ensembles

eff_mean = eff_stack.mean("ens")
eff_min = eff_stack.min("ens")
eff_max = eff_stack.max("ens")

time_rel = eff_mean.time_rel

# =========================================================
# --- FG & DIC SNAPSHOTS AT 180 h AFTER RELEASE ---
# =========================================================
# Cumulative fields from OAE_ensemble.ipynb §5 (t = 0 at the first air-sea flux anomaly)
groups = {"g1": "July", "g2": "November"}

FG_maps = {}
DIC_maps = {}

for g, season in groups.items():
    ds_cumsum = xr.open_dataset(intermediate(f"Iceland3_CDR_cstar_cumsum_fields_{season}.nc"))
    FG_maps[g] = ds_cumsum["FG_CO2_cumsum_anom"].sel(time_rel=180).mean("ens")
    DIC_maps[g] = ds_cumsum["hDIC_anom"].sel(time_rel=180).mean("ens")

lon = FG_maps["g1"]["lon"]
lat = FG_maps["g1"]["lat"]
land_mask = FG_maps["g1"] == 0

# =========================
# --- FIGURE LAYOUT ---
# =========================
fig = plt.figure(figsize=(12, 10))

gs = gridspec.GridSpec(
    nrows=3, ncols=2, height_ratios=[1, 1, 1], hspace=0.30, wspace=0.30
)

proj = ccrs.Mercator()
pc = ccrs.PlateCarree()


def plot_map(
    ax, data, title, label, letter, vmin=None, vmax=None, show_lat_labels=True
):
    ax.set_title(title, fontsize=12)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=pc)

    ax.contourf(
        lon,
        lat,
        land_mask,
        levels=[0.5, 1.5],
        colors=["lightgrey"],
        transform=pc,
        zorder=2,
    )

    im = ax.pcolormesh(
        lon,
        lat,
        data,
        transform=pc,
        shading="auto",
        vmin=vmin,
        vmax=vmax,
        zorder=1,
    )

    gl = ax.gridlines(
        crs=pc,
        draw_labels=True,
        linewidth=0.5,
        linestyle="--",
        alpha=0.5,
    )

    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = show_lat_labels
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    cb = plt.colorbar(im, ax=ax, shrink=0.6)
    cb.set_label(label)

    ax.text(
        -0.05,
        1.05,
        letter,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="top",
        ha="right",
    )


# =========================
# --- MAP PANELS ---
# =========================
ax_map1 = fig.add_subplot(gs[0, 0], projection=proj)
plot_map(
    ax_map1,
    FG_maps["g1"],
    "cumulative CO$_2$ flux anomaly at 180h\n (July)",
    "mmol m$^{-2}$",
    "a)",
    vmin=0,
    vmax=0.001 * 3600,
)

ax_map2 = fig.add_subplot(gs[0, 1], projection=proj)
plot_map(
    ax_map2,
    DIC_maps["g1"],
    "DIC integrated anomaly snapshot at 180h\n (July)",
    "mmol m$^{-2}$",
    "b)",
    vmin=0,
    vmax=1,
    show_lat_labels=False,
)

ax_map3 = fig.add_subplot(gs[1, 0], projection=proj)
plot_map(
    ax_map3,
    FG_maps["g2"],
    "cumulative CO$_2$ flux anomaly at 180h\n (November)",
    "mmol m$^{-2}$",
    "c)",
    vmin=0,
    vmax=0.001 * 3600,
)

ax_map4 = fig.add_subplot(gs[1, 1], projection=proj)
plot_map(
    ax_map4,
    DIC_maps["g2"],
    "DIC integrated anomaly snapshot at 180h\n (November)",
    "mmol m$^{-2}$",
    "d)",
    vmin=0,
    vmax=1,
    show_lat_labels=False,
)

# =========================
# --- TIME SERIES ---
# =========================
ax_ts = fig.add_subplot(gs[2, :])

ax_ts.fill_between(
    time_rel,
    eff_min.values,
    eff_max.values,
    alpha=0.3,
    label="Ensemble spread",
)

for ens in eff_stack.ens.values:
    ax_ts.plot(
        time_rel,
        eff_stack.sel(ens=ens).values,
        linewidth=1,
        alpha=0.9,
        linestyle="-",
        label=f"{ens}",
    )

ax_ts.plot(
    time_rel,
    eff_mean.values,
    linewidth=1.5,
    color="black",
    label="Ensemble mean",
)

# Reference lines at 0, 96, and 180 hrs
for t, ls, col in zip(
    [0, 96], [":", ":"], ["red", "red"]
):
    ax_ts.axvline(x=t, color=col, linestyle=ls, linewidth=1.8)

ax_ts.set_title(r"CDR efficiency ($\int \text{FG}_{\text{CO}_2} \, dt \ / \ \Delta \text{ALK}$)")
ax_ts.set_xlabel("Time (hours after release)")
ax_ts.set_ylabel("Efficiency")

ax_ts.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
ax_ts.legend(ncol=4, frameon=False)

ax_ts.text(
    -0.02,
    1.08,
    "e)",
    transform=ax_ts.transAxes,
    fontsize=16,
    fontweight="bold",
    va="top",
    ha="right",
)
plt.setp(ax_ts.get_xticklabels(), rotation=30, ha="right")

plt.subplots_adjust(top=0.95, bottom=0.08)
plt.show()

## 5. Release snapshots

Maps of upper Hvalfjörður 6, 12, 24 and 48 h after the release start, for every member:
- ALK anomaly at s_rho = 54;
- pH anomaly at s_rho = 54;
- surface pCO₂ anomaly.

Figures are saved to `fig_dir` as `Iceland3_Hvalfjordur_{ens}_{s_rho54, pH_s_rho54, surface_pCO2}_snapshots.png` and are not shown inline. This section moved here from `OAE_ensemble.ipynb`.

In [ ]:
proj = ccrs.Mercator(central_longitude=-21.45)
data_crs = ccrs.PlateCarree()
hvalfjordur_extent = [-21.55, -21.35, 64.345, 64.410]  # upper Hvalfjörður

snapshot_vars = {
    "ALK_anom": {
        "title": r"Alkalinity Anomaly ($\Delta$ALK)",
        "cbar_label": r"Alkalinity Anomaly $\Delta$ALK [meq/m$^3$]",
        "cmap": "YlOrRd",
        "vmin": 0.0,
        "vmax": 50.0,
        "at_level": True,
        "file_suffix": "s_rho54",
        "model_hour": "model_hour_bgc",
    },
    "pH_anom": {
        "title": r"pH Anomaly ($\Delta$pH)",
        "cbar_label": r"pH Anomaly [$\Delta$pH units]",
        "cmap": "YlOrRd",
        "vmin": 0.0,
        "vmax": 0.15,
        "at_level": True,
        "file_suffix": "pH_s_rho54",
        "model_hour": "model_hour_dia",
    },
    "pCO2SURF_anom": {
        "title": r"Surface $\text{pCO}_2$ Anomaly ($\Delta\text{pCO}_2$)",
        "cbar_label": r"Surface $\text{pCO}_2$ Anomaly [ppmv]",
        "cmap": "RdYlBu_r",  # negative values: CO2 drawdown from the alkalinity addition
        "vmin": -50.0,
        "vmax": 0.0,
        "at_level": False,
        "file_suffix": "surface_pCO2",
        "model_hour": "model_hour_dia",
    },
}


def add_scalebar_at(ax, lon, lat, length_km=2.0):
    """Draws a geodesic scale bar at the specified lat/lon anchor point."""
    R = 6371.0  # Earth radius in km
    dlon = (length_km / (R * np.cos(np.radians(lat)))) * (180.0 / np.pi)
    ax.plot([lon, lon + dlon], [lat, lat], color="black", linewidth=2.5, transform=data_crs, zorder=10)
    ax.text(
        lon + dlon / 2.0,
        lat + 0.0015,
        f"{length_km:.0f} km",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        transform=data_crs,
        zorder=10,
        bbox=dict(boxstyle="square,pad=0.15", facecolor="white", edgecolor="none", alpha=0.7),
    )


for season in seasons:
    ds_snap = xr.open_dataset(intermediate(f"Iceland3_release_snapshots_{season}.nc"))
    lon_rho = ds_snap["lon_rho"].values
    lon_rho_180 = np.where(lon_rho > 180, lon_rho - 360, lon_rho)
    lat_rho = ds_snap["lat_rho"].values
    mask_rho = ds_snap["mask_rho"].values  # 1 = ocean, 0 = land
    land_overlay = np.where(mask_rho == 0, 1.0, np.nan)
    s_rho_level = ds_snap.attrs["s_rho"]

    for ens in ds_snap["ens"].values:
        member = ds_snap.sel(ens=ens)
        release_start = float(member["release_start_hour"])

        for var, cfg in snapshot_vars.items():
            fig, axes = plt.subplots(2, 2, figsize=(14, 10), subplot_kw={"projection": proj})
            cbar_mesh = None

            for ax, hrs in zip(axes.flatten(), member["hours_after_release"].values):
                snapshot = member.sel(hours_after_release=hrs)
                anomaly_ocean = np.where(mask_rho == 1, snapshot[var].values, np.nan)

                ax.set_extent(hvalfjordur_extent, crs=data_crs)
                # Model land cells and coastline from mask_rho
                ax.pcolormesh(
                    lon_rho_180, lat_rho, land_overlay,
                    transform=data_crs, cmap="binary", vmin=0, vmax=1.5, shading="auto", zorder=2,
                )
                ax.contour(
                    lon_rho_180, lat_rho, mask_rho,
                    levels=[0.5], colors="black", linewidths=0.8, transform=data_crs, zorder=3,
                )
                cbar_mesh = ax.pcolormesh(
                    lon_rho_180, lat_rho, anomaly_ocean,
                    transform=data_crs, cmap=cfg["cmap"], vmin=cfg["vmin"], vmax=cfg["vmax"],
                    shading="auto", zorder=1,
                )

                gl = ax.gridlines(
                    crs=data_crs, draw_labels=True, linewidth=0.5, linestyle="--", color="gray", alpha=0.6
                )
                gl.top_labels = False
                gl.right_labels = False
                add_scalebar_at(ax, lon=-21.39, lat=64.350, length_km=2.0)

                model_hour = float(snapshot[cfg["model_hour"]])
                ax.set_title(f"+{hrs} Hours Post-Release (t = {model_hour:.1f}h)", fontsize=11, fontweight="bold")

            cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
            cb = fig.colorbar(cbar_mesh, cax=cbar_ax)
            cb.set_label(cfg["cbar_label"], fontsize=12, fontweight="bold")

            level = f" at s_rho={s_rho_level}" if cfg["at_level"] else ""
            plt.suptitle(
                rf"Iceland3 MARBL - {ens}: {cfg['title']}{level}"
                f"\nUpper Hvalfjörður Snapshots (Release Start: t = {release_start:.0f}h)",
                fontsize=14,
                fontweight="bold",
                y=0.98,
            )

            out_name = os.path.join(fig_dir, f"Iceland3_Hvalfjordur_{ens}_{cfg['file_suffix']}_snapshots.png")
            plt.savefig(out_name, dpi=300, bbox_inches="tight")
            plt.close(fig)
            print(f"Saved {out_name}")

    ds_snap.close()

## Appendix: diagnostics

- Flux-based CDR efficiency and cumulative CO₂ uptake of every member in absolute time.
- Time coverage of the CDR-metric and regridded cstar files.
- The CDR forcing of CDR3. This last cell reads a model **input** file from `/anvil`, not an intermediate file.

In [ ]:
import os
import matplotlib.pyplot as plt
import xarray as xr

# =========================
# Paths & File Setup
# =========================
files = {ens: intermediate(f"{ens}_cdr_metrics.nc") for ens in ensembles}

data = {}

# =========================
# Load Pre-Calculated Data
# =========================
for ens, path in files.items():
    ds = xr.open_dataset(path)

    eff = ds["cdr_efficiency"].load()
    uptake = ds["co2_uptake"].load()
    time_abs = ds["time"].load().values

    data[ens] = {
        "time_abs": time_abs,
        "cdr_efficiency": eff.values,
        "co2_uptake": uptake.values,
    }
    ds.close()

# =========================
# Plotting - one panel per member (Absolute Time)
# =========================
fig, axes = plt.subplots(len(ensembles), 1, figsize=(12, 3.5 * len(ensembles)), sharex=False)

color_eff = "tab:blue"
color_uptake = "tab:green"

for i, ens in enumerate(ensembles):
    ax = axes[i]

    if ens not in data:
        ax.set_title(f"{ens}: Data File Missing", fontsize=11, fontweight="bold", color="red")
        continue

    t_abs = data[ens]["time_abs"]
    eff = data[ens]["cdr_efficiency"]
    uptake = data[ens]["co2_uptake"]

    # Efficiency (Left Y-Axis)
    ax.plot(t_abs, eff, color=color_eff, linewidth=2, label="CDR Efficiency")
    ax.set_ylabel("CDR Efficiency", color=color_eff, fontweight="bold")
    ax.tick_params(axis="y", labelcolor=color_eff)
    ax.grid(True, linestyle="--", alpha=0.5)

    # CO2 Uptake (Right Y-Axis)
    ax_twin = ax.twinx()
    ax_twin.plot(t_abs, uptake, color=color_uptake, linewidth=2, linestyle="--", label="CO$_2$ Uptake")
    ax_twin.set_ylabel("Cumulative CO$_2$ Uptake", color=color_uptake, fontweight="bold")
    ax_twin.tick_params(axis="y", labelcolor=color_uptake)

    ax.set_title(f"{ens}: Flux-based CDR Metrics (Absolute Time)", fontsize=11, fontweight="bold")
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right")

axes[-1].set_xlabel("Date / Time", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "CDR_metrics_absolute_time.jpeg"), format="jpeg", dpi=300)
plt.show()

In [ ]:
import os
import xarray as xr

# =========================
# File Paths Configuration
# =========================
metrics_paths = {ens: os.path.join(intermediate_dir, f"{ens}_cdr_metrics.nc") for ens in ensembles}
regridded_paths = {
    ens: os.path.join(intermediate_dir, f"ds_cstar{ens.removeprefix('CDR')}_regridded.nc")
    for ens in ensembles
}

# Print Header
print(f"{'Ensemble':<8} | {'Dataset Type':<18} | {'Steps':<8} | {'Start Time':<20} | {'End Time':<20} | {'Duration (hrs)':<14}")
print("-" * 95)

for ens in ensembles:
    m_path = metrics_paths.get(ens)
    r_path = regridded_paths.get(ens)
    
    # --- Check Metrics File ---
    if os.path.exists(m_path):
        with xr.open_dataset(m_path) as ds_m:
            time_m = ds_m["time"]
            len_m = len(time_m)
            start_m = str(time_m.values[0])[:19]
            end_m = str(time_m.values[-1])[:19]
            
            dt_m = time_m.isel(time=-1) - time_m.isel(time=0)
            dur_m = float(dt_m.dt.total_seconds() / 3600.0) if hasattr(dt_m, "dt") else float(dt_m.values / 3600.0)
            
            print(f"{ens:<8} | {'cdr_metrics.nc':<18} | {len_m:<8} | {start_m:<20} | {end_m:<20} | {dur_m:<14.2f}")
    else:
        print(f"{ens:<8} | {'cdr_metrics.nc':<18} | Missing File")

    # --- Check Regridded File ---
    if os.path.exists(r_path):
        with xr.open_dataset(r_path) as ds_r:
            time_r = ds_r["time"]
            len_r = len(time_r)
            start_r = str(time_r.values[0])[:19]
            end_r = str(time_r.values[-1])[:19]
            
            dt_r = time_r.isel(time=-1) - time_r.isel(time=0)
            dur_r = float(dt_r.dt.total_seconds() / 3600.0) if hasattr(dt_r, "dt") else float(dt_r.values / 3600.0)
            
            print(f"{'':<8} | {'regridded.nc':<18} | {len_r:<8} | {start_r:<20} | {end_r:<20} | {dur_r:<14.2f}")
    else:
        print(f"{'':<8} | {'regridded.nc':<18} | Missing File")
        
    print("-" * 95)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# File path
file_path = "/anvil/projects/x-ees250129/x-uheede/Iceland_experiments/Iceland3_MARBL_2024_CDR3/Iceland3_cdr_forcing_ENS3.nc"

# Open dataset
ds = xr.open_dataset(file_path)

# 1. Decode tracer_name character array into clean string list
raw_tracer_names = ds["tracer_name"].values
if raw_tracer_names.ndim == 2:
    tracer_names = [
        "".join(
            c.decode("utf-8") if isinstance(c, bytes) else str(c) for c in row
        ).strip()
        for row in raw_tracer_names
    ]
else:
    tracer_names = [
        s.decode("utf-8").strip() if isinstance(s, bytes) else str(s).strip()
        for s in raw_tracer_names
    ]

# Extract time and convert to hours relative to release start/file start
time_vals = ds["time"].values
if np.issubdtype(time_vals.dtype, np.datetime64):
    time_hours = (time_vals - time_vals[0]) / np.timedelta64(1, "h")
else:
    time_hours = (
        time_vals - time_vals[0]
    ) * 24.0  # Convert days to hours if numeric

# Extract volume and tracer data (squeezing out single ncdr dimension)
cdr_vol = ds["cdr_volume"].squeeze().values  # Shape: (time,)
cdr_tr = ds["cdr_tracer"].squeeze().values  # Shape: (time, ntracers)

# Find indices of tracers that have non-zero data
active_tracer_indices = [
    i for i in range(len(tracer_names)) if np.any(np.nan_to_num(cdr_tr[:, i]) > 0)
]

# 2. Plotting - 2 Panel Figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Panel 1: CDR Volume Flux
ax1.plot(time_hours, cdr_vol, marker="o", color="black", linewidth=2)
ax1.set_ylabel("CDR Volume Flux (m³/s)", fontsize=11, fontweight="bold")
ax1.set_title("CDR Forcing: Volume Flux & Tracer Concentrations Over Time", fontsize=12, fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.5)

# Panel 2: CDR Tracer Concentrations
if active_tracer_indices:
    for idx in active_tracer_indices:
        t_name = tracer_names[idx]
        ax2.plot(
            time_hours,
            cdr_tr[:, idx],
            marker="s",
            label=f"Tracer: {t_name}",
            linewidth=1.8,
        )
    ax2.legend(loc="upper right", frameon=True)
else:
    # If no non-zero tracers, plot ALK and ALK_ALT_CO2 explicitly
    for name in ["ALK", "ALK_ALT_CO2"]:
        if name in tracer_names:
            idx = tracer_names.index(name)
            ax2.plot(
                time_hours,
                cdr_tr[:, idx],
                marker="s",
                label=f"Tracer: {name}",
                linewidth=1.8,
            )
    ax2.legend(loc="upper right", frameon=True)

ax2.set_xlabel("Time (hours since file start)", fontsize=11)
ax2.set_ylabel("Tracer Concentration", fontsize=11, fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

ds.close()

## Notes and changes from the original notebook

The original version of this notebook is kept outside the repository, in `archive/notebooks/oae_analysis.ipynb`.

**To do** (notes kept from the original):
- Plot sum of ALK over time
- Slice ALK anomaly with depth

**Changes**
- *File reading:*
  - Every input is read from `intermediate_dir` through `intermediate()`, which stops with a clear message if a file is missing. Previously some cells skipped missing files silently.
  - Seasonal files `…_July.nc` / `…_November.nc` replace the no-suffix / `_pt2` files.
  - Variables are named: `pCO2SURF_anom`, `pH`, `ALK_anom`, `DIC_anom`, `max_pH`.
- *CDR metrics:* §1 and §4 read `CDR{n}_cdr_metrics.nc`. The old §4 cell read `Iceland3_CDR{n}_cdr_metrics.nc`, which no current cell writes. Variable-name fallbacks were removed.
- *§3:* loads the July extrema explicitly, instead of using variables left over from earlier cells.
- *§4:* the maps read the cumulative fields written by `OAE_ensemble.ipynb` §5 instead of recomputing them here. t = 0 is now the first air–sea flux anomaly; previously it was the first domain-mean DIC anomaly > 1e-8.
- *§5:* the release snapshot maps moved here from `OAE_ensemble.ipynb`.
- *Figures:* the absolute-time CDR metrics figure shows all six members. Figures are saved to `fig_dir`; two were previously saved into the intermediate-files folder.
- *Removed:*
  - the stale configuration and imports cells;
  - the cell that loaded files into unreachable `globals()` names;
  - the unused cumulative-fields load and one-line inspection cells;
  - the cell using undefined `DIC_anom` / `dz_da`;
  - the older CDR1–3 version of the §4 figure, which read from `/anvil/projects/.../Intermediate_files/`;
  - empty cells.